# Create Custom ReAct Pattern

### Here we will manually create ReAct agent.

**Steps for agent:**

- Create tools
- Bind tools with llm
- call llm_with_tools
- If llm has responded with tool_call detail. Then, Manually call tool. Otherwise, return response of llm
- Update message history with tool results
- After getting tool response, call llm again
- return response to user

`NOTE: We can also use langchain ReAct Agents (create_agent) for making this flow automatic. For this please check: 1.agents-and-tools.ipynb`

In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini")

## Creating Custom Tools with LangChain

In [3]:
@tool
def add(a: int, b: int) -> int:
    """
    Add a and b.
    
    Args:
        a (int): first integer to be added
        b (int): second integer to be added

    Return:
        int: sum of a and b
    """
    return a + b

@tool
def subtract(a: int, b:int) -> int:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: int, b:int) -> int:
    """Multiply a and b."""
    return a * b

In [4]:
tool_map = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply
}

input_ = { "a": 1, "b": 2 }

tool_map["add"].invoke(input_)

3

## Bind tools with llm

In [5]:
tools = [add, subtract, multiply]
llm_with_tools = llm.bind_tools(tools)

## Interacting with the Model

In [10]:
query = "What is 3 + 2?"
chat_history = [HumanMessage(content=query)] # chat_history array will contain the entire conversation between user and LLM

### Invoke LLM with query

In [ ]:
response_1 = llm_with_tools.invoke(chat_history)
chat_history.append(response_1)

print(type(response_1))
response_1 # Here is AI response.

<class 'langchain_core.messages.ai.AIMessage'>


AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 132, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-fb07ced6-efb7-499c-9df9-0078d913c8e9-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'type': 'tool_call'}], usage_metadata={'input_tokens': 132, 'output_tokens': 17, 'total_tokens': 149})

`llm has responded with instruction to call a tool and it has provided arguments and tool_name`

In [12]:
chat_history

[HumanMessage(content='What is 3 + 2?'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 132, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-fb07ced6-efb7-499c-9df9-0078d913c8e9-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'type': 'tool_call'}], usage_metadata={'input_tokens': 132, 'output_tokens': 17, 'total_tokens': 149})]

### Parse tool calls

`NOTE: We are only fetching 1st tool_call here. `

If llm responds with multiple tool calls, we are not covering this scenario.

In [13]:
tool_calls_1 = response_1.tool_calls

tool_1_name = tool_calls_1[0]["name"]
tool_1_args = tool_calls_1[0]["args"]
tool_call_1_id = tool_calls_1[0]["id"]

print(f'tool name --> {tool_1_name}')
print(f'tool args --> {tool_1_args}')
print(f'tool call ID --> {tool_call_1_id}')

tool name --> add
tool args --> {'a': 3, 'b': 2}
tool call ID --> call_YdzKb4PToK5QIeF6InGNj253


### Invoke the Tool

In [14]:
tool_response = tool_map[tool_1_name].invoke(tool_1_args)
tool_message = ToolMessage(content=tool_response, tool_call_id=tool_call_1_id)

print(tool_message)

content='5' tool_call_id='call_YdzKb4PToK5QIeF6InGNj253'


In [ ]:
chat_history.append(tool_message) # modifying chat_history

### Generate a final answer from chat history

In [ ]:
answer = llm_with_tools.invoke(chat_history)
chat_history.append(answer)

print(type(answer))
print(answer.content)

<class 'langchain_core.messages.ai.AIMessage'>
3 + 2 equals 5.


In [17]:
answer

AIMessage(content='3 + 2 equals 5.', response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 157, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'stop', 'logprobs': None}, id='run-fa554b0d-d340-4e43-956b-a5e71a92607f-0', usage_metadata={'input_tokens': 157, 'output_tokens': 9, 'total_tokens': 166})

In [19]:
chat_history

[HumanMessage(content='What is 3 + 2?'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 132, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-fb07ced6-efb7-499c-9df9-0078d913c8e9-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 2}, 'id': 'call_YdzKb4PToK5QIeF6InGNj253', 'type': 'tool_call'}], usage_metadata={'input_tokens': 132, 'output_tokens': 17, 'total_tokens': 149}),
 ToolMessage(content='5', tool_call_id='call_YdzKb4PToK5QIeF6InGNj253'),
 AIMessage(content='3 +

## Building an Agent

#### Combine all of the above functionality in one class

In [20]:
class ToolCallingAgent:
    def __init__(self, llm):
        self.llm_with_tools = llm.bind_tools(tools)
        self.tool_map = tool_map

    def run(self, query: str) -> str:
        # Step 1: Initial user message
        chat_history = [HumanMessage(content=query)]

        # Step 2: LLM chooses tool
        response = self.llm_with_tools.invoke(chat_history)
        if not response.tool_calls:
            return response.contet # Direct response, no tool needed
        # Step 3: Handle first tool call
        tool_call = response.tool_calls[0]
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # Step 4: Call first tool manually
        tool_result = self.tool_map[tool_name].invoke(tool_args)

        # Step 5: Send result back to LLM
        tool_message = ToolMessage(content=str(tool_result), tool_call_id=tool_call_id)
        chat_history.extend([response, tool_message]) # Here we haven't yet appended final AI Response.
        print("---------------- CHAT_HISTORY_COMBINED ------------------")
        print(chat_history)
        print("------------ CHAT_HISTORY_SEPARATED ---------------")
        for msg in chat_history:
            if msg.type == "ai" and msg.tool_calls:
                print(f"AI (Tool Call): {msg.tool_calls}")
            else:
                print(f"{msg.type.upper()}: {msg.content}")


        # Step 6: Final LLM result
        final_response = self.llm_with_tools.invoke(chat_history)
        return final_response.content


In [21]:
agent = ToolCallingAgent(llm)
response = agent.run("multiply 3 with 5")
print("response: ", response)

---------------- CHAT_HISTORY_COMBINED ------------------
[HumanMessage(content='multiply 3 with 5'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_9PaMvuoEKqgKi31RGe3PXBOl', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 130, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-0f1ed501-0c39-4eae-95c2-c60077db5b6b-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_9PaMvuoEKqgKi31RGe3PXBOl', 'type': 'tool_call'}], usage_metadata={'input_tokens': 130, 'output_tokens': 17, 'total_tokens': 147}), ToolMessage(content='15', 

In [22]:
agent = ToolCallingAgent(llm)
response = agent.run("multiply 3 with 5 and then add response with 4")

---------------- CHAT_HISTORY_COMBINED ------------------
[HumanMessage(content='multiply 3 with 5 and then add response with 4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_07YO7EUAx93PMChHrV9oGGz5', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 137, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_373a14eb6f', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-07a70b7c-1444-4ac6-aa28-ee16efe095d0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'call_07YO7EUAx93PMChHrV9oGGz5', 'type': 'tool_call'}], usage_metadata={'input_tokens': 137, 'output_tokens': 17, 'total_tokens': 154}

In [24]:
response

''

`response` in above is `null`. Because, llm requires here multi step reasoning. Means: it needs to call another tool after 1st tool response.

**NOTE:**

This agent can handle only one step tool calling process.\
`MEANS:` If llm wants to run more tools based on last response, then we are not covering this scenario.